# Credit Card Fraud Detection

**Goal:** Detect fraudulent transactions (highly imbalanced data)
**Algorithm:** Random Forest + Undersampling
**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve)
%matplotlib inline

In [ ]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


## 1. Load Data from Kaggle

In [ ]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(f"{path}/creditcard.csv")
print ('Shape: %s' % (df.shape,))
print ('Columns: V1-V28 (anonymized), Time, Amount, Class')

<hr>## 2. Exploratory Data Analysis

In [ ]:
fraud_count = df['Class'].sum()
total = len(df)
print ('Total transactions: %d' % total)
print ('Fraud cases: %d (%.4f%%)' % (fraud_count, fraud_count/total*100))
print ('Normal cases: %d' % (total - fraud_count))
print ('Fraud ratio: 1 : %d' % ((total-fraud_count)//fraud_count))

In [ ]:
# Visualize the imbalance
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(x='Class', data=df)
plt.title('Class Distribution (0=Normal, 1=Fraud)')
plt.xticks([0, 1], ['Normal', 'Fraud'])

plt.subplot(1, 2, 2)
plt.hist(df['Amount'][df['Class']==0], bins=50, alpha=0.7, label='Normal', density=True)
plt.hist(df['Amount'][df['Class']==1], bins=50, alpha=0.7, label='Fraud', density=True)
plt.xlabel('Transaction Amount')
plt.ylabel('Density')
plt.legend()
plt.title('Amount Distribution by Class')

plt.tight_layout()
plt.show()

<hr>## 3. Train/Test Split (before resampling)

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print ('Train: %d, Test: %d' % (X_train.shape[0], X_test.shape[0]))
print ('Train fraud: %d, Test fraud: %d' % (y_train.sum(), y_test.sum()))

<hr>## 4. Handle Imbalance (Undersampling)

In [ ]:
fraud_train = X_train[y_train == 1]
normal_train = X_train[y_train == 0]
n_fraud = len(fraud_train)

# Take equal number of normal samples
normal_sample = normal_train.sample(n=n_fraud, random_state=42)
X_balanced = pd.concat([fraud_train, normal_sample])
y_balanced = np.array([1] * n_fraud + [0] * n_fraud)

print ('Before undersampling: %d normal, %d fraud' % (len(normal_train), n_fraud))
print ('After undersampling:  %d samples (50%% fraud, 50%% normal)' % len(X_balanced))

<hr>## 5. Train Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_balanced, y_balanced)
print ('Model: %s' % model)

<hr>## 6. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print ('ROC-AUC Score: %.4f' % roc_auc_score(y_test, y_prob))
print ('\nConfusion Matrix:\n%s' % confusion_matrix(y_test, y_pred))
print ('\nClassification Report:')
print (classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label='ROC (AUC = %.4f)' % roc_auc_score(y_test, y_prob))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

<hr>## 7. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print ('Top 10 most important features (anonymized V features):')
print (importances.head(10).to_string(index=False))